# 07 — Corporate Climate Credibility Score (CCCS) Dashboard

**Project:** Corporate Climate Credibility Audit — power-sector Proof-of-Concept (POC)

**Author:** Souvik Mandal (drsouvikmandal@gmail.com)

**Notebook role.** This notebook produces a single self-contained HTML file (`dashboard/index.html`) that lets a reviewer interactively explore the Corporate Climate Credibility Score (CCCS) framework. The dashboard has three tiers:

- **Macro tier** — the headline 2×2 quadrant chart with three reactive sliders that let the user adjust the (Pledge Quality Score, EPA Performance Score, Satellite Cross-Validation Score) weights and watch the leaderboard reorder live.
- **Drilldown tier** — a parent picker that surfaces the selected parent's component scores, 2011–2023 EPA Greenhouse Gas Reporting Program (GHGRP) emissions trajectory, quadrant assignment, and a "Robust Credible Leader" / "Robust Laggard" badge when applicable.
- **Audit tier** — a sortable table showing each scored parent's rank under all six sensitivity-analysis weight schemes (the same six defined in notebook 06 §6), with min/max rank and stability flags.

The HTML embeds plotly.js inline (≈ 3.5 MB) so the dashboard works offline. A reviewer can open `dashboard/index.html` in any modern browser by double-clicking the file — no server, no installation, no internet required.

## What this notebook delivers

| Section | What it produces |
|---|---|
| 1 | Load the four inputs from notebooks 01 and 06 |
| 2 | Build the embedded JSON data blobs (per-parent components, rank stability, per-parent GHGRP trajectories) |
| 3 | Build the dashboard HTML (HTML structure + CSS + JavaScript) |
| 4 | Write `dashboard/index.html` and report file size |
| 5 | How to view + summary |

## Significance of this notebook

The headline 2×2 quadrant chart and the per-parent scores are valuable static artifacts (notebook 06's outputs). But a reviewer evaluating the framework wants to *interrogate* it:

- *"What if I cared more about Talk and less about Verify? Does my preferred parent still rank well?"*
- *"For a specific parent — say Vistra — how did their emissions trajectory actually look, and what's their credibility breakdown across the three axes?"*
- *"Which parents are robust top-quartile performers under any reasonable weighting scheme, and which are weight-dependent?"*

A static figure can't answer those interactively. A self-contained HTML dashboard can. The reviewer opens it in their browser, moves sliders, picks parents, sorts tables — exactly the kind of exploration that builds (or erodes) trust in the framework's claims.

The dashboard is also the most defensible answer to the HBS rubric criterion *"Judgment in choosing appropriate methods, NOT just using everything"*: the framework's default weights (0.2 / 0.4 / 0.4) are a *choice*, and the dashboard makes that choice explicit and challengeable rather than hidden inside a CSV.

## Architectural decision — vanilla JavaScript, not a server framework

We use vanilla JavaScript + Plotly.js (not Plotly Dash or Streamlit) for the interactive behavior. The reason is reviewer accessibility: a single `.html` file that opens by double-click has zero friction. A Dash app requires `pip install dash`, running `python app.py`, managing ports, and keeping a terminal open — none of which is reasonable to ask of an HBS reviewer evaluating a take-home submission. Vanilla JS gives us identical functionality (sliders, dynamic tables, reactive charts) with no server.

## Slider behavior — normalize on every change

The three weight sliders (PQS, EPA Performance, SCVS) can be moved independently. On every change, the dashboard reads the three raw slider values, normalizes them to sum to 1.0, and uses the normalized weights to recompute the Corporate Climate Credibility Score (CCCS) for every parent. The displayed weights below each slider show the *effective* (normalized) weight in [0, 1].

This is simpler than a sequential-locking UX (where moving one slider auto-adjusts the others) and delivers the same functional outcome — the user can reach any valid weight combination on the unit simplex. A "Reset to default (0.2 / 0.4 / 0.4)" button restores the workplan-locked default weights.

## What changes when the user moves a slider

**Important methodological note:** the chart's spatial layout does *not* change with weight movement. Position (PQS × EPA Performance), color (SCVS), and size (2023 emissions) all depend on per-parent component scores that are *inputs* to CCCS, not derived from CCCS itself. Quadrant assignment (median splits on PQS × EPA Performance) is similarly weight-independent — quadrant is a structural classification based on the components, not on the composite.

What *does* change:

- **Leaderboard table** — reorders by new CCCS values
- **Hover tooltips on the chart** — show the parent's CCCS under the current weights
- **Drilldown panel** — selected parent's CCCS rank updates

This is the methodologically correct behavior: the user is exploring how *the weighting choice affects credibility ranking*, not how it affects the underlying components.

## 0. Environment

In [1]:
from __future__ import annotations
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Plotly.js inline bundle for the dashboard's offline-capable HTML
try:
    import plotly.offline as plotly_offline
except ImportError:
    import subprocess; subprocess.check_call(["pip", "install", "-q", "plotly"])
    import plotly.offline as plotly_offline

PROJECT_ROOT   = Path("/Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DASHBOARD_DIR  = PROJECT_ROOT / "dashboard"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

# Default workplan-locked weights for CCCS composite
W_DEFAULT = {"pqs": 0.20, "eps": 0.40, "scvs": 0.40}

print(f"Project root:      {PROJECT_ROOT}")
print(f"Dashboard output:  {DASHBOARD_DIR}")
print(f"Default weights:   PQS={W_DEFAULT['pqs']}  EPA={W_DEFAULT['eps']}  SCVS={W_DEFAULT['scvs']}")

Project root:      /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task
Dashboard output:  /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/dashboard
Default weights:   PQS=0.2  EPA=0.4  SCVS=0.4


## 1. Load inputs

Four CSVs from upstream notebooks:

| File | Producer | Used for |
|---|---|---|
| `cohort_cccs.csv` | Notebook 06 §7 | All 50 cohort parents with their CCCS scores, quadrant labels, and raw components |
| `cohort_cccs_components.csv` | Notebook 06 §7 | Slim per-parent component scores for dashboard live-recompute |
| `cohort_cccs_rank_stability.csv` | Notebook 06 §7 | Per-parent rank under all 6 sensitivity weight schemes (for the Audit tier) |
| `cohort_parent_year_panel.csv` | Notebook 01 | Per-parent yearly GHGRP emissions (for the Drilldown trajectory chart) |

In [2]:
cohort_cccs       = pd.read_csv(DATA_PROCESSED / "cohort_cccs.csv")
cohort_components = pd.read_csv(DATA_PROCESSED / "cohort_cccs_components.csv")
rank_stability    = pd.read_csv(DATA_PROCESSED / "cohort_cccs_rank_stability.csv")
parent_year_panel = pd.read_csv(DATA_PROCESSED / "cohort_parent_year_panel.csv")

print(f"cohort_cccs:                  {cohort_cccs.shape}  cols={cohort_cccs.columns.tolist()[:6]}...")
print(f"cohort_cccs_components:       {cohort_components.shape}")
print(f"cohort_cccs_rank_stability:   {rank_stability.shape}")
print(f"cohort_parent_year_panel:     {parent_year_panel.shape}")

# Quick coverage summary
print(f"\nCoverage:")
print(f"  Total cohort parents:                 {len(cohort_components)}")
print(f"  PQS-scored (Talk-NA excluded):        {cohort_components['pqs_composite'].notna().sum()}")
print(f"  Full CCCS (all 3 axes; sensitivity):  {len(rank_stability)}")
print(f"  Parents with trajectory data:         {parent_year_panel['parent_name'].nunique()}")

cohort_cccs:                  (50, 13)  cols=['parent_name', 'rank', 'attributed_co2e', 'quadrant', 'pqs_composite', 'eps_score']...
cohort_cccs_components:       (50, 6)
cohort_cccs_rank_stability:   (24, 22)
cohort_parent_year_panel:     (650, 5)

Coverage:
  Total cohort parents:                 50
  PQS-scored (Talk-NA excluded):        26
  Full CCCS (all 3 axes; sensitivity):  24
  Parents with trajectory data:         50


## 2. Build embedded JSON data blobs

The dashboard's JavaScript needs three data structures embedded directly in the HTML (no fetch calls, no external files):

- **`COHORT_DATA`** — array of 50 parents, each with name, attributed 2023 emissions, components (PQS / EPS / SCVS), default CCCS, default quadrant.
- **`RANK_STABILITY`** — object keyed by parent name, with per-scheme rank and the `always_top_quartile` / `always_bottom_quartile` flags from notebook 06 §6.
- **`PARENT_TRAJECTORIES`** — object keyed by parent name, with the per-year (2011–2023) GHGRP attributed CO₂e values for the Drilldown trajectory mini-chart.

We serialize all three as `json.dumps(..., default=float)` so numpy/pandas types convert cleanly. NaN values become `null` in JSON, which JavaScript handles natively.

In [3]:
def _safe_float(x):
    """Convert numpy/pandas numeric to plain Python float; NaN -> None for JSON."""
    if pd.isna(x): return None
    return float(x)


# ---------- COHORT_DATA: 50-element array of parent records ----------
cohort_data = []
for _, r in cohort_cccs.iterrows():
    cohort_data.append({
        "name":        str(r["parent_name"]),
        "rank":        int(r["rank"]),
        "emissions_mt": _safe_float(r["attributed_co2e"]) / 1e6 if pd.notna(r["attributed_co2e"]) else None,
        "quadrant":    str(r["quadrant"]),
        "pqs":         _safe_float(r["pqs_composite"]),
        "eps":         _safe_float(r["eps_score"]),
        "scvs":        _safe_float(r["scvs_score"]),
        "cccs_default":  _safe_float(r["cccs_composite"]),
        "cccs_partial":  _safe_float(r["cccs_partial_composite"]),
    })

# ---------- RANK_STABILITY: keyed by parent ----------
SCHEMES = ["default", "equal", "action_heavy", "talk_heavy", "walk_heavy", "verify_heavy"]
rank_stability_dict = {}
for _, r in rank_stability.iterrows():
    name = str(r["parent_name"])
    rank_stability_dict[name] = {
        "ranks": {s: int(r[f"rank_{s}"]) for s in SCHEMES},
        "scores": {s: _safe_float(r[f"cccs_{s}"]) for s in SCHEMES},
        "min_rank":   int(r["min_rank"]),
        "max_rank":   int(r["max_rank"]),
        "rank_range": int(r["rank_range"]),
        "always_top_quartile":    bool(r["always_top_quartile"]),
        "always_bottom_quartile": bool(r["always_bottom_quartile"]),
    }

# ---------- PARENT_TRAJECTORIES: keyed by parent ----------
parent_trajectories = {}
for name, group in parent_year_panel.groupby("parent_name"):
    g = group.sort_values("year")
    parent_trajectories[str(name)] = {
        "years":     g["year"].tolist(),
        "emissions": [_safe_float(v) / 1e6 if pd.notna(v) else None  for v in g["attributed_co2e"]],
    }

# Serialize
COHORT_JSON       = json.dumps(cohort_data, indent=0)
RANK_STAB_JSON    = json.dumps(rank_stability_dict, indent=0)
TRAJECTORIES_JSON = json.dumps(parent_trajectories, indent=0)

print(f"COHORT_DATA:         {len(cohort_data)} parents, JSON size = {len(COHORT_JSON):,} bytes")
print(f"RANK_STABILITY:      {len(rank_stability_dict)} fully-scored parents, JSON size = {len(RANK_STAB_JSON):,} bytes")
print(f"PARENT_TRAJECTORIES: {len(parent_trajectories)} parents, JSON size = {len(TRAJECTORIES_JSON):,} bytes")
print(f"Total embedded data: {(len(COHORT_JSON)+len(RANK_STAB_JSON)+len(TRAJECTORIES_JSON)):,} bytes")

COHORT_DATA:         50 parents, JSON size = 11,077 bytes
RANK_STABILITY:      24 fully-scored parents, JSON size = 11,020 bytes
PARENT_TRAJECTORIES: 50 parents, JSON size = 17,643 bytes
Total embedded data: 39,740 bytes


## 3. Build the dashboard HTML

Assemble the full HTML: header + CSS + Plotly.js inline bundle + embedded JSON + JavaScript logic + section markup (Macro / Drilldown / Audit). All in one self-contained string.

In [4]:
# Get the inline plotly.js bundle (~3.5 MB)
PLOTLY_JS_INLINE = plotly_offline.get_plotlyjs()
print(f"Plotly.js bundle: {len(PLOTLY_JS_INLINE)/1e6:.2f} MB")

# Cohort summary stats for the header
n_total      = len(cohort_components)
n_pqs_scored = int(cohort_components["pqs_composite"].notna().sum())
n_talk_na    = n_total - n_pqs_scored
n_full_cccs  = len(rank_stability)

# CSS — minimal, professional, no framework
CSS = """
* { box-sizing: border-box; }
body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
       margin: 0; padding: 24px; color: #222; background: #fafafa; max-width: 1400px; margin: 0 auto; }
h1 { font-size: 24px; margin-bottom: 4px; }
h2 { font-size: 20px; margin-top: 32px; border-bottom: 2px solid #ddd; padding-bottom: 6px; }
h3 { font-size: 15px; margin: 12px 0 6px 0; }
.subtitle { color: #555; font-size: 14px; margin-top: 0; }
.macro-row { display: flex; gap: 16px; margin-top: 12px; }
.macro-chart { flex: 1.6; }
.macro-leaderboard { flex: 1; min-width: 340px; }
table { border-collapse: collapse; width: 100%; font-size: 12px; }
th, td { padding: 5px 8px; text-align: left; border-bottom: 1px solid #eee; }
th { background: #f0f0f0; font-weight: 600; cursor: pointer; user-select: none; }
th:hover { background: #e0e0e0; }
td.num { text-align: right; font-variant-numeric: tabular-nums; }
tr.top { background: #eaf7e6; } tr.top:hover { background: #d4f0cd; }
tr.bot { background: #fbeaea; } tr.bot:hover { background: #f4d4d4; }
.slider-group { display: grid; grid-template-columns: 270px 1fr 60px; gap: 12px; align-items: center;
                margin: 6px 0; max-width: 800px; }
.slider-group label { font-weight: 600; }
input[type=range] { width: 100%; }
button { padding: 6px 14px; font-size: 13px; cursor: pointer;
         border: 1px solid #888; background: #fff; border-radius: 4px; }
button:hover { background: #eee; }
.weight-display { font-variant-numeric: tabular-nums; color: #444; font-weight: 600; }
.drilldown-row { display: flex; gap: 16px; margin-top: 12px; }
.parent-detail-text { margin-top: 12px; padding: 10px; background: #f5f5f5; border-radius: 4px; font-size: 13px; }
.parent-detail-text .badge { display: inline-block; padding: 3px 8px; border-radius: 3px; font-weight: 600; margin-top: 4px; }
.badge-leader  { background: #2e7d32; color: white; }
.badge-laggard { background: #c62828; color: white; }
.audit-wrap { overflow-x: auto; }
footer { margin-top: 40px; padding-top: 16px; border-top: 1px solid #ddd; font-size: 12px; color: #666; }

/* Responsive sizing for Plotly charts and flex layouts */
.macro-row > *, .drilldown-row > * { min-width: 0; }
#quadrant-chart, #parent-bars, #parent-trajectory { width: 100%; }
.parent-name-header { margin: 12px 0 4px 0; font-size: 16px; color: #333; }
@media (max-width: 900px) {
  .macro-row, .drilldown-row { flex-direction: column; }
}
"""

# Header section (uses f-string for stat insertion)
HEADER_HTML = f"""
<header>
  <h1>Corporate Climate Credibility Score (CCCS) Dashboard</h1>
  <p class="subtitle">Power-sector POC — top {n_total} U.S. fossil-fuel power-generating parents by 2023 EPA GHGRP attributed CO₂-equivalent emissions</p>
  <p class="subtitle"><strong>Built by:</strong> Souvik Mandal, Email: <a href="mailto:drsouvikmandal@gmail.com">drsouvikmandal@gmail.com</a>. Find the full project on <a href="https://github.com/Souvik-Mandal-Harvard/climate-ghgrp-data-science" target="_blank">GitHub</a>.</p>
  <p class="subtitle"><strong>Coverage:</strong> {n_pqs_scored} parents with Pledge Quality Score (Talk axis) — {n_talk_na} Talk-NA (no public disclosure source). {n_full_cccs} parents have full CCCS (all three axes) and are eligible for the sensitivity ranking in the Audit tier below.</p>
</header>
"""

# Macro section
MACRO_HTML = """
<section id="macro">
  <h2>1. Macro — 2×2 Credibility Quadrant</h2>
  <p>Dot position = (Pledge Quality Score, EPA Performance Score). Color = Satellite Cross-Validation Score (Climate TRACE vs GHGRP agreement). Size = 2023 emissions. Hover any dot for full per-parent detail. Move the three weight sliders below to see how the CCCS leaderboard re-ranks under alternative weightings — the chart's position/color/size encoding stays fixed (those are inputs, not outputs of the weighting choice).</p>

  <div class="macro-row">
    <div class="macro-chart" id="quadrant-chart"></div>
    <div class="macro-leaderboard">
      <h3>Live CCCS Leaderboard</h3>
      <table id="leaderboard-table">
        <thead><tr><th>#</th><th>Parent</th><th class="num">CCCS</th><th>Quadrant</th></tr></thead>
        <tbody></tbody>
      </table>
    </div>
  </div>

  <h3 style="margin-top:16px;">Weight controls</h3>
  <div class="slider-group">
    <label for="slider-pqs">Pledge Quality Score:</label>
    <input type="range" id="slider-pqs"  min="0" max="1" step="0.01" value="0.20" oninput="onSliderChange()">
    <span class="weight-display" id="weight-pqs-display">0.20</span>
  </div>
  <div class="slider-group">
    <label for="slider-eps">EPA Performance Score:</label>
    <input type="range" id="slider-eps"  min="0" max="1" step="0.01" value="0.40" oninput="onSliderChange()">
    <span class="weight-display" id="weight-eps-display">0.40</span>
  </div>
  <div class="slider-group">
    <label for="slider-scvs">Satellite Cross-Validation Score:</label>
    <input type="range" id="slider-scvs" min="0" max="1" step="0.01" value="0.40" oninput="onSliderChange()">
    <span class="weight-display" id="weight-scvs-display">0.40</span>
  </div>
  <div style="margin-top:8px;"><button onclick="resetWeights()">Reset to default (0.20 / 0.40 / 0.40)</button>
    <span style="margin-left:16px; color:#666; font-size:12px;">Displayed weights are normalized so PQS + EPS + SCVS = 1.</span></div>
</section>
"""

# Drilldown section
DRILLDOWN_HTML = """
<section id="drilldown">
  <h2>2. Drilldown — Per-parent inspection</h2>
  <p>Pick a parent to see their three-axis component breakdown, 2011–2023 EPA GHGRP emissions trajectory, quadrant assignment, and rank-stability badge (if any).</p>

  <div style="margin: 12px 0;">
    <label for="parent-picker"><strong>Parent Company: </strong></label>
    <select id="parent-picker" onchange="onParentChange(this.value)" style="font-size:14px; padding:4px;">
      <option value="">— select a parent —</option>
    </select>
  </div>

  <div id="parent-detail" style="display:none;">
    <h3 class="parent-name-header" id="parent-name-display"></h3>
    <div class="parent-detail-text" id="parent-text"></div>
    <div class="drilldown-row">
      <div id="parent-bars" style="flex:11;"></div>
      <div id="parent-trajectory" style="flex:9;"></div>
    </div>
  </div>
</section>
"""

# Audit section
AUDIT_HTML = """
<section id="audit">
  <h2>3. Audit — Per-parent rank under six weight schemes</h2>
  <p>For each of the 24 fully-scored parents (those with all three axes present), this table shows their CCCS rank under each of the six sensitivity-analysis weight schemes from notebook 06 §6: <em>default</em> (0.2 / 0.4 / 0.4), <em>equal</em> (1/3 / 1/3 / 1/3), <em>action-heavy</em> (0.1 / 0.45 / 0.45), <em>talk-heavy</em> (0.4 / 0.3 / 0.3), <em>walk-heavy</em> (0.2 / 0.6 / 0.2), <em>verify-heavy</em> (0.2 / 0.2 / 0.6). Click any column header to sort. Robust top-quartile and robust bottom-quartile flags identify parents whose classification doesn't depend on the weighting choice.</p>

  <div class="audit-wrap">
    <table id="audit-table">
      <thead><tr>
        <th onclick="sortAuditTable(0)">Parent</th>
        <th onclick="sortAuditTable(1)" class="num">PQS</th>
        <th onclick="sortAuditTable(2)" class="num">EPS</th>
        <th onclick="sortAuditTable(3)" class="num">SCVS</th>
        <th onclick="sortAuditTable(4)" class="num">Default</th>
        <th onclick="sortAuditTable(5)" class="num">Equal</th>
        <th onclick="sortAuditTable(6)" class="num">Action</th>
        <th onclick="sortAuditTable(7)" class="num">Talk</th>
        <th onclick="sortAuditTable(8)" class="num">Walk</th>
        <th onclick="sortAuditTable(9)" class="num">Verify</th>
        <th onclick="sortAuditTable(10)" class="num">min</th>
        <th onclick="sortAuditTable(11)" class="num">max</th>
        <th onclick="sortAuditTable(12)" class="num">range</th>
        <th onclick="sortAuditTable(13)">Robust?</th>
      </tr></thead>
      <tbody></tbody>
    </table>
  </div>
</section>
"""

FOOTER_HTML = """
<footer>
  <p><strong>Data sources</strong>: EPA GHGRP (2011–2023 power-sector facility-year emissions), EPA AMPD CEMS (continuous stack measurements), Climate TRACE v5.6.0 (asset-level satellite-derived emissions estimates), SBTi target dashboard, Net Zero Tracker, U.S. SEC EDGAR 10-K filings.</p>
  <p><strong>Methodology</strong>: see notebook 03 (statistical modeling of emissions-data reliability), notebook 05 (Pledge Quality Score via dual-LLM scoring), notebook 06 (CCCS composite construction + 6-scheme sensitivity).</p>
  <p><strong>Citation note</strong>: 74% S&P-500 restatement rate documented by Cohen, Rouen &amp; Sachdeva (2026), <em>Nature Climate Change</em>, 16, 33–36, DOI 10.1038/s41558-025-02494-9; HBS BiGS coverage December 2025.</p>
</footer>
"""

# JS data block (f-string-interpolated to inject the JSON)
JS_DATA_BLOCK = f"""
const COHORT_DATA = {COHORT_JSON};
const RANK_STABILITY = {RANK_STAB_JSON};
const PARENT_TRAJECTORIES = {TRAJECTORIES_JSON};
const W_DEFAULT = {{pqs: 0.20, eps: 0.40, scvs: 0.40}};
const PQS_MEDIAN = {float(cohort_components["pqs_composite"].median()):.4f};
const EPS_MEDIAN = 50.0;
"""

# JS logic block (plain string — no f-string, raw JavaScript)
JS_LOGIC_BLOCK = r"""
// ============ CCCS RECOMPUTE ============
function recomputeCCCS(pqs, eps, scvs, w_pqs, w_eps, w_scvs) {
  if (pqs == null || eps == null || scvs == null) return null;
  const pqs_f = Math.max(pqs, 1.0);
  const eps_f = Math.max(eps, 1.0);
  const scvs_f = Math.max(scvs, 1.0);
  return Math.exp(w_pqs * Math.log(pqs_f) + w_eps * Math.log(eps_f) + w_scvs * Math.log(scvs_f));
}

function getNormalizedWeights() {
  let p = parseFloat(document.getElementById('slider-pqs').value);
  let e = parseFloat(document.getElementById('slider-eps').value);
  let s = parseFloat(document.getElementById('slider-scvs').value);
  const sum = p + e + s;
  if (sum < 1e-9) { return {pqs: 1/3, eps: 1/3, scvs: 1/3}; }
  return {pqs: p/sum, eps: e/sum, scvs: s/sum};
}

// ============ LEADERBOARD ============
function rebuildLeaderboard(parents_sorted) {
  const tbody = document.querySelector('#leaderboard-table tbody');
  tbody.innerHTML = '';
  // Top 10
  parents_sorted.slice(0, 10).forEach((p, i) => {
    const row = document.createElement('tr');
    row.className = 'top';
    row.innerHTML = '<td>' + (i+1) + '</td>' +
                    '<td>' + p.name + '</td>' +
                    '<td class="num">' + p.cccs_current.toFixed(1) + '</td>' +
                    '<td>' + p.quadrant + '</td>';
    tbody.appendChild(row);
  });
  // Separator
  const sep = document.createElement('tr');
  sep.innerHTML = '<td colspan="4" style="text-align:center; color:#888; font-style:italic; padding:6px;">. . .</td>';
  tbody.appendChild(sep);
  // Bottom 5
  const bot5 = parents_sorted.slice(-5);
  bot5.forEach((p, i) => {
    const rank = parents_sorted.length - bot5.length + i + 1;
    const row = document.createElement('tr');
    row.className = 'bot';
    row.innerHTML = '<td>' + rank + '</td>' +
                    '<td>' + p.name + '</td>' +
                    '<td class="num">' + p.cccs_current.toFixed(1) + '</td>' +
                    '<td>' + p.quadrant + '</td>';
    tbody.appendChild(row);
  });
}

// ============ CHART (positions/colors/sizes fixed; only hovers update) ============
let SCORED_PARENTS = null;  // cache of PQS-scored parents (for the right panel)
let TALK_NA_PARENTS = null; // cache of Talk-NA parents (for the left panel)

function initializeChart() {
  SCORED_PARENTS  = COHORT_DATA.filter(p => p.pqs != null);
  TALK_NA_PARENTS = COHORT_DATA.filter(p => p.pqs == null);

  // Marker size mapping (area ∝ emissions)
  const emissions = COHORT_DATA.map(p => p.emissions_mt || 0);
  const e_min = Math.min(...emissions), e_max = Math.max(...emissions);
  function sizeFor(em) {
    if (em == null) return 10;
    return 10 + 60 * (em - e_min) / (e_max - e_min);
  }
  SCORED_PARENTS.forEach(p => p._size = sizeFor(p.emissions_mt));
  TALK_NA_PARENTS.forEach(p => p._size = sizeFor(p.emissions_mt));

  // Compute initial CCCS using default weights
  const w = W_DEFAULT;
  SCORED_PARENTS.forEach(p => {
    p.cccs_current = recomputeCCCS(p.pqs, p.eps, p.scvs, w.pqs, w.eps, w.scvs);
  });
  TALK_NA_PARENTS.forEach(p => p.cccs_current = null);

  const hoverScored = "<b>%{customdata[0]}</b><br>" +
                      "PQS:  %{customdata[1]:.1f}<br>" +
                      "EPS:  %{customdata[2]:.1f}<br>" +
                      "SCVS: %{customdata[3]:.1f}<br>" +
                      "Quadrant: %{customdata[4]}<br>" +
                      "2023 emissions: %{customdata[5]:.2f} MtCO₂e<br>" +
                      "<b>CCCS (current weights): %{customdata[6]:.1f}</b>" +
                      "<extra></extra>";
  const hoverNA = "<b>%{customdata[0]}</b><br>" +
                  "PQS: Talk-NA (no public disclosure)<br>" +
                  "EPS:  %{customdata[2]:.1f}<br>" +
                  "SCVS: %{customdata[3]:.1f}<br>" +
                  "2023 emissions: %{customdata[5]:.2f} MtCO₂e" +
                  "<extra></extra>";

  function customData(arr, includeCCCS) {
    return arr.map(p => [p.name, p.pqs, p.eps, p.scvs, p.quadrant, p.emissions_mt,
                          includeCCCS ? (p.cccs_current != null ? p.cccs_current : null) : null]);
  }

  const traceNA = {
    type: 'scatter', mode: 'markers',
    xaxis: 'x', yaxis: 'y',
    x: TALK_NA_PARENTS.map(_ => 0), y: TALK_NA_PARENTS.map(p => p.eps),
    marker: { size: TALK_NA_PARENTS.map(p => p._size),
              color: TALK_NA_PARENTS.map(p => p.scvs), colorscale: 'Viridis', cmin: 0, cmax: 100,
              showscale: false, line: {width: 0.5, color: 'black'}, opacity: 0.85 },
    customdata: customData(TALK_NA_PARENTS, false), hovertemplate: hoverNA,
    showlegend: false, name: 'Talk-NA',
  };
  const traceScored = {
    type: 'scatter', mode: 'markers',
    xaxis: 'x2', yaxis: 'y2',
    x: SCORED_PARENTS.map(p => p.pqs), y: SCORED_PARENTS.map(p => p.eps),
    marker: { size: SCORED_PARENTS.map(p => p._size),
              color: SCORED_PARENTS.map(p => p.scvs), colorscale: 'Viridis', cmin: 0, cmax: 100,
              showscale: true,
              colorbar: { title: {text: 'SCVS<br>(100 = best<br>CT-vs-GHGRP<br>agreement)', side: 'right'},
                          len: 0.8, thickness: 14, xpad: 6 },
              line: {width: 0.5, color: 'black'}, opacity: 0.85 },
    customdata: customData(SCORED_PARENTS, true), hovertemplate: hoverScored,
    showlegend: false, name: 'Scored',
  };

  const layout = {
    grid: { rows: 1, columns: 2, pattern: 'independent', xgap: 0.04 },
    // Left (Talk-NA) panel — narrow strip, no y-axis decorations (scale is mirrored on yaxis2)
    xaxis:  {
      title: { text: 'Talk-NA<br>(no PQS)', font: { size: 11 } },
      range: [-0.8, 0.8], showticklabels: false, domain: [0, 0.08]
    },
    yaxis:  {
      range: [-5, 105],
      showticklabels: false, showline: false, showgrid: false, zeroline: false
    },
    // Right (main 2×2) panel — owns the visible y-axis labels and title via yaxis2
    xaxis2: {
      title: { text: 'Pledge Quality Score (PQS) — 0 to 100', font: { size: 13 } },
      range: [-5, 105], domain: [0.18, 0.95]
    },
    yaxis2: {
      title: { text: 'EPA Performance Score (EPS) — 0 to 100 (100 = fastest decarbonization)',
               font: { size: 13 }, standoff: 8 },
      range: [-5, 105], matches: 'y', anchor: 'x2', showticklabels: true
    },
    shapes: [
      // EPS=50 horizontal line on left (Talk-NA) panel — references yaxis (the hidden left y-axis)
      { type: 'line', xref: 'x',  yref: 'y',  x0: -0.8, x1: 0.8, y0: 50, y1: 50,
        line: { color: 'gray', width: 1, dash: 'dot' } },
      // EPS=50 horizontal line on right (main) panel — now references yaxis2
      { type: 'line', xref: 'x2', yref: 'y2', x0: -5, x1: 105, y0: 50, y1: 50,
        line: { color: 'gray', width: 1, dash: 'dot' } },
      // PQS median vertical line on right panel — also references yaxis2 now
      { type: 'line', xref: 'x2', yref: 'y2', x0: PQS_MEDIAN, x1: PQS_MEDIAN, y0: -5, y1: 105,
        line: { color: 'gray', width: 1.2, dash: 'dot' } },
    ],
    annotations: [
      // All four quadrant labels live on the right panel — yref switched from 'y' to 'y2'
      { xref: 'x2', yref: 'y2', x: 97, y: 97, text: '<b>Credible Leaders</b>',
        xanchor: 'right', yanchor: 'top',    showarrow: false, font: { size: 11, color: 'rgba(0,0,0,0.45)' } },
      { xref: 'x2', yref: 'y2', x: 3,  y: 97, text: '<b>Quiet Achievers</b>',
        xanchor: 'left',  yanchor: 'top',    showarrow: false, font: { size: 11, color: 'rgba(0,0,0,0.45)' } },
      { xref: 'x2', yref: 'y2', x: 97, y: 3,  text: '<b>Greenwashers</b>',
        xanchor: 'right', yanchor: 'bottom', showarrow: false, font: { size: 11, color: 'rgba(0,0,0,0.45)' } },
      { xref: 'x2', yref: 'y2', x: 3,  y: 3,  text: '<b>Laggards</b>',
        xanchor: 'left',  yanchor: 'bottom', showarrow: false, font: { size: 11, color: 'rgba(0,0,0,0.45)' } },
    ],
    autosize: true, height: 580, margin: { t: 30, b: 80, l: 60, r: 140 },
    hovermode: 'closest', plot_bgcolor: 'white',
  };

  Plotly.newPlot('quadrant-chart', [traceNA, traceScored], layout, {responsive: true});
}

function updateChartHovers() {
  function customData(arr) {
    return arr.map(p => [p.name, p.pqs, p.eps, p.scvs, p.quadrant, p.emissions_mt,
                          p.cccs_current != null ? p.cccs_current : null]);
  }
  Plotly.restyle('quadrant-chart', { customdata: [customData(SCORED_PARENTS)] }, [1]);
}

// ============ SLIDER + RESET ============
function onSliderChange() {
  const w = getNormalizedWeights();
  document.getElementById('weight-pqs-display').textContent = w.pqs.toFixed(2);
  document.getElementById('weight-eps-display').textContent = w.eps.toFixed(2);
  document.getElementById('weight-scvs-display').textContent = w.scvs.toFixed(2);

  // Recompute CCCS for all scored parents
  SCORED_PARENTS.forEach(p => {
    p.cccs_current = recomputeCCCS(p.pqs, p.eps, p.scvs, w.pqs, w.eps, w.scvs);
  });
  // Sort + rank
  const sorted = SCORED_PARENTS.filter(p => p.cccs_current != null)
                                .slice().sort((a, b) => b.cccs_current - a.cccs_current);
  sorted.forEach((p, i) => { p.cccs_rank_current = i + 1; });

  // Update UI
  rebuildLeaderboard(sorted);
  updateChartHovers();
  const sel = document.getElementById('parent-picker').value;
  if (sel) updateDrilldownRank(sel);
}

function resetWeights() {
  document.getElementById('slider-pqs').value = 0.20;
  document.getElementById('slider-eps').value = 0.40;
  document.getElementById('slider-scvs').value = 0.40;
  onSliderChange();
}

// ============ DRILLDOWN ============
function initializeDrilldown() {
  const picker = document.getElementById('parent-picker');
  // Sort by rank for predictable dropdown order
  const sorted = COHORT_DATA.slice().sort((a, b) => a.rank - b.rank);
  sorted.forEach(p => {
    const opt = document.createElement('option');
    opt.value = p.name;
    opt.textContent = '#' + p.rank + ' — ' + p.name;
    picker.appendChild(opt);
  });
}

function onParentChange(name) {
  if (!name) {
    document.getElementById('parent-detail').style.display = 'none';
    return;
  }
  const p = COHORT_DATA.find(x => x.name === name);
  if (!p) return;

  // Update the parent-name header above the charts
  document.getElementById('parent-name-display').textContent = '#' + p.rank + '  —  ' + p.name;

  // CRITICAL: show the container BEFORE Plotly.newPlot so the charts can measure
  // their container width. Plotly's autosize requires a visible (non-display:none)
  // parent div; otherwise it falls back to a default width (~700px) that overflows
  // narrow flex containers.
  document.getElementById('parent-detail').style.display = 'block';

  // Three-axis bar chart
  Plotly.newPlot('parent-bars', [{
    type: 'bar', orientation: 'h',
    x: [p.pqs, p.eps, p.scvs],
    y: ['Pledge Quality (PQS)', 'EPA Performance (EPS)', 'Satellite Cross-Val (SCVS)'],
    marker: { color: ['#3949ab', '#ff7043', '#43a047'] },
    text: [(p.pqs != null ? p.pqs.toFixed(1) : 'NA'), p.eps.toFixed(1), p.scvs.toFixed(1)],
    textposition: 'outside',
  }], {
    title: { text: 'Three-axis component breakdown', font: {size: 13} },
    xaxis: { range: [0, 110], title: 'Score (0-100)' },
    autosize: true, margin: { t: 40, l: 180, r: 60, b: 40 }, height: 280,
  }, {responsive: true});

  // Trajectory chart
  const traj = PARENT_TRAJECTORIES[name];
  if (traj) {
    Plotly.newPlot('parent-trajectory', [{
      x: traj.years, y: traj.emissions, type: 'scatter', mode: 'lines+markers',
      line: { color: '#d32f2f', width: 2 }, marker: { size: 7 },
    }], {
      title: { text: 'EPA GHGRP CO₂e (Mt) — 2011-2023', font: {size: 13} },
      xaxis: { title: 'Year', tickformat: 'd' },
      yaxis: { title: 'Mt CO₂-equivalent', rangemode: 'tozero' },
      autosize: true, margin: { t: 40, l: 60, r: 40, b: 50 }, height: 280,
    }, {responsive: true});
  } else {
    document.getElementById('parent-trajectory').innerHTML = '<p style="color:#888; padding:20px;">No trajectory data for this parent.</p>';
  }

  // Text details
  const w = getNormalizedWeights();
  const cccs = recomputeCCCS(p.pqs, p.eps, p.scvs, w.pqs, w.eps, w.scvs);
  const cccsText = cccs != null ? cccs.toFixed(1) : 'NaN (PQS unavailable for Talk-NA parents)';
  const stab = RANK_STABILITY[name];
  let badge = '';
  if (stab) {
    if (stab.always_top_quartile) {
      badge = '<span class="badge badge-leader">🌟 Robust Credible Leader (top quartile under all 6 weight schemes)</span>';
    } else if (stab.always_bottom_quartile) {
      badge = '<span class="badge badge-laggard">⚠️ Robust Laggard (bottom quartile under all 6 weight schemes)</span>';
    } else {
      badge = '<span style="color:#666; font-style:italic;">Rank stable across schemes: range ' +
              stab.min_rank + '-' + stab.max_rank + '</span>';
    }
  } else {
    badge = '<span style="color:#888; font-style:italic;">Not in sensitivity sample (missing one or more axes).</span>';
  }
  let rankText = '';
  if (cccs != null) {
    const sorted = COHORT_DATA.filter(x => x.cccs_current != null)
                              .slice().sort((a,b) => b.cccs_current - a.cccs_current);
    const idx = sorted.findIndex(x => x.name === name);
    rankText = (idx + 1) + ' of ' + sorted.length + ' fully-scored';
  } else {
    rankText = '— (no full CCCS rank for this parent)';
  }
  document.getElementById('parent-text').innerHTML =
    '<strong>Cohort rank (by 2023 emissions): #' + p.rank + ' of 50</strong>  •  ' +
    '<strong>Quadrant: ' + p.quadrant + '</strong>  •  ' +
    '<strong>CCCS (current weights):</strong> ' + cccsText + '  •  ' +
    '<strong>CCCS rank:</strong> ' + rankText + '<br>' + badge;

  // Belt-and-suspenders: force Plotly to re-measure container widths now that
  // the layout pass triggered by display:block has fully completed. Catches any
  // edge case where the initial newPlot rendered before the browser finished
  // computing the new flex-child widths.
  Plotly.Plots.resize('parent-bars');
  if (traj) Plotly.Plots.resize('parent-trajectory');
}

function updateDrilldownRank(name) {
  onParentChange(name);  // simplest: re-render the whole drilldown
}

// ============ AUDIT TABLE ============
const SCHEMES = ['default', 'equal', 'action_heavy', 'talk_heavy', 'walk_heavy', 'verify_heavy'];

function initializeAuditTable() {
  const tbody = document.querySelector('#audit-table tbody');
  tbody.innerHTML = '';
  Object.entries(RANK_STABILITY).forEach(([name, d]) => {
    const p = COHORT_DATA.find(x => x.name === name);
    const robust = d.always_top_quartile ? '🌟 Top'
                  : d.always_bottom_quartile ? '⚠️ Bottom'
                  : '';
    const row = document.createElement('tr');
    row.innerHTML =
      '<td>' + name + '</td>' +
      '<td class="num">' + (p ? p.pqs.toFixed(1) : '—') + '</td>' +
      '<td class="num">' + (p ? p.eps.toFixed(1) : '—') + '</td>' +
      '<td class="num">' + (p ? p.scvs.toFixed(1) : '—') + '</td>' +
      SCHEMES.map(s => '<td class="num">' + d.ranks[s] + '</td>').join('') +
      '<td class="num">' + d.min_rank + '</td>' +
      '<td class="num">' + d.max_rank + '</td>' +
      '<td class="num">' + d.rank_range + '</td>' +
      '<td>' + robust + '</td>';
    tbody.appendChild(row);
  });
}

function sortAuditTable(colIdx) {
  const tbody = document.querySelector('#audit-table tbody');
  const rows = Array.from(tbody.querySelectorAll('tr'));
  const currentCol = tbody.dataset.sortCol;
  const currentDir = tbody.dataset.sortDir || 'asc';
  const newDir = (String(colIdx) === currentCol && currentDir === 'asc') ? 'desc' : 'asc';

  rows.sort((a, b) => {
    const va = a.children[colIdx].textContent.trim();
    const vb = b.children[colIdx].textContent.trim();
    const na = parseFloat(va), nb = parseFloat(vb);
    if (!isNaN(na) && !isNaN(nb)) {
      return newDir === 'asc' ? na - nb : nb - na;
    }
    return newDir === 'asc' ? va.localeCompare(vb) : vb.localeCompare(va);
  });

  tbody.innerHTML = '';
  rows.forEach(r => tbody.appendChild(r));
  tbody.dataset.sortCol = colIdx;
  tbody.dataset.sortDir = newDir;
}

// ============ INIT ============
window.addEventListener('DOMContentLoaded', () => {
  initializeChart();
  initializeDrilldown();
  initializeAuditTable();
  onSliderChange();  // initial leaderboard population at default weights
});
"""

# Assemble the full HTML
HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Corporate Climate Credibility Score (CCCS) Dashboard</title>
  <style>{CSS}</style>
  <script>{PLOTLY_JS_INLINE}</script>
</head>
<body>
{HEADER_HTML}
{MACRO_HTML}
{DRILLDOWN_HTML}
{AUDIT_HTML}
{FOOTER_HTML}
<script>
{JS_DATA_BLOCK}
{JS_LOGIC_BLOCK}
</script>
</body>
</html>
"""

print(f"Total HTML size: {len(HTML)/1e6:.2f} MB ({len(HTML):,} chars)")

Plotly.js bundle: 4.84 MB
Total HTML size: 4.91 MB (4,907,551 chars)


## 4. Write `dashboard/index.html`

In [5]:
out_path = DASHBOARD_DIR / "index.html"
out_path.write_text(HTML, encoding="utf-8")
size_mb = out_path.stat().st_size / 1e6
print(f"Wrote: {out_path}")
print(f"File size: {size_mb:.2f} MB")
print(f"\nTo view: double-click the file (or open in any modern browser).")
print(f"  file://{out_path.as_posix()}")

Wrote: /Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/dashboard/index.html
File size: 4.91 MB

To view: double-click the file (or open in any modern browser).
  file:///Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task/dashboard/index.html


## 5. How to view the dashboard

1. **Open `dashboard/index.html`** in any modern browser (Chrome, Firefox, Safari, Edge) by double-clicking the file. No installation, no server, no internet required — plotly.js is bundled inline.
2. **Interact:**
   - **Macro tier**: drag the three sliders (PQS, EPS, SCVS). The leaderboard reorders live; the chart's hover tooltips show each parent's CCCS under your current weighting. Click "Reset to default" to return to the workplan-locked 0.20 / 0.40 / 0.40.
   - **Drilldown tier**: pick a parent from the dropdown to see their three-axis breakdown, GHGRP emissions trajectory, and rank-stability badge.
   - **Audit tier**: click any column header to sort. Look for the 🌟 and ⚠️ flags to find parents whose classification is weight-independent.

## Summary

The dashboard ties together three previously-independent analytical surfaces (notebook 05's Pledge Quality scoring, notebook 01's EPA emissions trajectories, notebook 03's Climate TRACE vs GHGRP LME diagnostic) into one interactive artifact. It exposes the framework's most defensible methodological commitment — *that credibility is a function of three independent signals, and the weighting choice is a deliberate, explicit, user-controllable parameter* — and lets a reviewer test how robust the headline rankings are to alternative weightings.

### Submission bundle

The take-home deliverable consists of:

1. **Notebook 03** (`03_Statistical_Modeling.ipynb`) — the statistical-modeling showcase: CEMS-vs-GHGRP validation panel + three-model arc Diagnostic LME with the headline finding that Climate TRACE adds little independent emissions signal once CEMS gross load is controlled for.
2. **Notebook 06** (`06_CCCS_Composite.ipynb`) — the composite construction + 2×2 quadrant assignment + 6-scheme sensitivity analysis. Produces `cohort_cccs.csv` as the per-parent scoreboard.
3. **This notebook 07** (`07_Dashboard.ipynb`) — produces `dashboard/index.html`, the interactive artifact for reviewer exploration.

Supporting notebooks 00, 01, 02, 05 (data acquisition, EDA, entity resolution, Pledge Quality Score) are upstream of the headline deliverables and run end-to-end to reproduce the inputs.

A complete dependency list is in `requirements.txt` at the project root.